# Accelerated EMRI Search: PSO + Gradient-Based Parameter Recovery

This notebook demonstrates the new accelerated search approach that starts from
a single STFT-identified frequency track and uses:

1. **Plunge-time estimation** from the observed `f/ḟ` ratio
2. **Particle Swarm Optimization** (PSO) for global parameter-space search
3. **Gradient-based refinement** via `TrackOptimizer` (Nelder-Mead / Adam)

We compare convergence speed and accuracy against the existing DE + Adam pipeline.

---

## Contents
1. Setup and data generation
2. STFT track extraction — what the search sees
3. Plunge-time estimation
4. `TrackOptimizer`: multi-start Nelder-Mead (FEW backend)
5. `ParticleSwarmOptimizer`: global PSO search
6. Comparison: new vs existing methods
7. (Optional) Gradient-based refinement with `fewtrax`


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp

jax.config.update('jax_enable_x64', True)

from emrisearch.search_utils import generate_emri_signal_and_sfts
from emrisearch.emri_utils import get_f_fdot_fddot_back
from emrisearch.jax_utils import det_stat as jax_det_stat
from emrisearch.track_optimizer import (
    TrackOptimizer, estimate_plunge_time, get_default_mode_candidates
)
from emrisearch.pso_utils import ParticleSwarmOptimizer

print('JAX devices:', jax.devices())


## 1. Setup and Data Generation

Generate a synthetic EMRI injection: signal + noise → SFTs.

In [ ]:
# True EMRI parameters: [M, mu, a, T_plunge, e_f, x0]
true_values = np.array([1e6, 10.0, 0.5, 0.2, 0.1, 1.0])

T_data  = 0.2    # years
T_sft   = 5e4    # seconds (~14 h)
deltaT  = 5.0    # seconds
snr_ref = 30.0

injection = generate_emri_signal_and_sfts(
    true_values, T_data, T_sft, deltaT, snr_ref, T_data
)

data_sfts  = jnp.asarray(injection['data_sfts'])
t_obs      = injection['t_alpha']           # SFT mid-times [s]
num_sfts   = injection['num_sfts']

print(f'SFT shape : {data_sfts.shape}  ({num_sfts} segments)')
print(f'Final SNR : {injection["snr_final"]:.1f}')


## 2. True STFT Track

Extract the true frequency track from the injection parameters — this is what
an ideal STFT detector would observe.

In [ ]:
phi, f, dotf, dotdotf = injection['true_phi_f_fdot_fddot']

# Dominant harmonic m=2, n=0
m, n = 2, 0
f_obs    = m * f[0]    + n * f[1]     # Hz
fdot_obs = m * dotf[0] + n * dotf[1]  # Hz/s

# Detection statistic with true parameters (upper bound)
f_alpha_jax    = jnp.asarray(f_obs)
fdot_alpha_jax = jnp.asarray(fdot_obs)
stat_true = float(jax_det_stat(data_sfts, f_alpha_jax, fdot_alpha_jax, T_sft=T_sft))
print(f'Detection statistic (true params, m={m}, n={n}): {stat_true:.2f}')

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
t_days = t_obs / 86400.0
axes[0].plot(t_days, f_obs * 1e3, 'C0.-')
axes[0].set_ylabel('f [mHz]')
axes[0].set_title(f'True EMRI track (m={m}, n={n})')
axes[0].grid(True)

axes[1].plot(t_days, fdot_obs * 1e12, 'C1.-')
axes[1].set_ylabel('ḟ [pHz/s]')
axes[1].set_xlabel('Time [days]')
axes[1].grid(True)

plt.tight_layout()
plt.show()


## 3. Plunge-Time Estimation

From the observed `(f, ḟ)` track, estimate the time to plunge using
the Peters chirp-time approximation.

In [ ]:
T_plunge_est, T_plunge_per_seg = estimate_plunge_time(f_obs, fdot_obs)

print(f'True T_plunge   : {true_values[3]:.4f} yr')
print(f'Estimated       : {T_plunge_est:.4f} yr')
print(f'Error           : {abs(T_plunge_est - true_values[3])*365.25:.1f} days')

plt.figure(figsize=(8, 3))
valid = fdot_obs > 0
plt.plot(t_days[valid], T_plunge_per_seg[valid] * 365.25, 'C0.', label='per-segment estimate')
plt.axhline(T_plunge_est * 365.25,  color='C1', ls='--', label=f'weighted mean = {T_plunge_est*365.25:.1f} days')
plt.axhline(true_values[3] * 365.25, color='k',  ls=':',  label=f'truth = {true_values[3]*365.25:.1f} days')
plt.xlabel('Time [days]')
plt.ylabel('T_plunge estimate [days]')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## 4. TrackOptimizer: Multi-Start Nelder-Mead

Uses the FEW trajectory backend (numpy) and minimizes the sum of squared
frequency residuals. The T_plunge bounds are narrowed using the estimate above.

In [ ]:
import time

optimizer = TrackOptimizer(f_obs, fdot_obs, t_obs, T_sft=T_sft)
print(f'Narrowed T_plunge bounds: [{optimizer.bounds[3,0]:.3f}, {optimizer.bounds[3,1]:.3f}] yr')

# Optimize for the true mode (2, 0) with 16 starts
t0 = time.perf_counter()
best_params, best_loss, all_results = optimizer.optimize(
    mode=(m, n), n_starts=16, max_iter=200
)
opt_time = time.perf_counter() - t0

print(f'\nOptimization time  : {opt_time:.1f} s')
print(f'Best track loss    : {best_loss:.3e} Hz²')
print(f'Best params found  : M={best_params[0]:.3e}, mu={best_params[1]:.2f}, '
      f'a={best_params[2]:.3f}, Tpl={best_params[3]:.4f}, ef={best_params[4]:.4f}')
print(f'True params        : M={true_values[0]:.3e}, mu={true_values[1]:.2f}, '
      f'a={true_values[2]:.3f}, Tpl={true_values[3]:.4f}, ef={true_values[4]:.4f}')


In [ ]:
# Evaluate detection statistic at found parameters
best_6d = np.append(best_params, 1.0)[np.newaxis, :]
_, f_best, fdot_best, _ = get_f_fdot_fddot_back(best_6d, t_obs)
f_alpha_best    = jnp.asarray(m * f_best[0]    + n * f_best[1])
fdot_alpha_best = jnp.asarray(m * fdot_best[0] + n * fdot_best[1])
stat_best = float(jax_det_stat(data_sfts, f_alpha_best, fdot_alpha_best, T_sft=T_sft))

print(f'Detection statistic at recovered params : {stat_best:.2f}')
print(f'Detection statistic at true params      : {stat_true:.2f}')
print(f'Recovery ratio                          : {stat_best/stat_true*100:.1f}%')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_days, f_obs * 1e3, 'C0.', label='observed (true)')
ax.plot(t_days, np.array(f_alpha_best) * 1e3, 'C1--', lw=2, label=f'recovered (stat={stat_best:.1f})')
ax.set_xlabel('Time [days]')
ax.set_ylabel('f [mHz]')
ax.set_title('TrackOptimizer: recovered vs observed track')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()


## 5. ParticleSwarmOptimizer: Global PSO Search

Initializes a swarm focused around the T_plunge estimate and maximizes
the detection statistic.

In [ ]:
pso = ParticleSwarmOptimizer(data_sfts, t_obs, T_sft=T_sft)

t0 = time.perf_counter()
pso_params, pso_fitness, history = pso.optimize(
    mode=(m, n),
    n_particles=50,          # use 50 for quick demo; 200 recommended
    n_iter=20,               # use 20 for quick demo; 100 recommended
    T_plunge_estimate=T_plunge_est,
    verbose=True,
)
pso_time = time.perf_counter() - t0

print(f'\nPSO time    : {pso_time:.1f} s')
print(f'Best stat   : {-pso_fitness:.2f}')
print(f'Best params : M={pso_params[0]:.3e}, mu={pso_params[1]:.2f}, '
      f'a={pso_params[2]:.3f}, Tpl={pso_params[3]:.4f}, ef={pso_params[4]:.4f}')


In [ ]:
# Plot PSO convergence history
plt.figure(figsize=(8, 4))
plt.plot(-np.array(history), 'C0.-')
plt.axhline(stat_true, color='k', ls=':', label=f'truth = {stat_true:.1f}')
plt.xlabel('PSO iteration')
plt.ylabel('Global best detection statistic')
plt.title('PSO convergence')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## 6. Comparison: New vs Existing Methods

Measure the detection statistic recovery as a function of computation time
for:
- **TrackOptimizer** (multi-start Nelder-Mead, FEW backend)
- **ParticleSwarmOptimizer** (PSO, FEW backend)
- **Baseline**: single random draw (no optimization)

In [ ]:
from emrisearch.draw_population import ParameterScaler

# Baseline: random parameter draws, no optimization
scaler   = ParameterScaler(optimizer.bounds, seed=0)
n_draws  = 20
rand_params = scaler.draw_samples(n_draws)

stats_random = []
for p in rand_params:
    try:
        p6 = np.append(p, 1.0)[np.newaxis, :]
        _, f_, fdot_, _ = get_f_fdot_fddot_back(p6, t_obs)
        fa  = jnp.asarray(m * f_[0]    + n * f_[1])
        fda = jnp.asarray(m * fdot_[0] + n * fdot_[1])
        stats_random.append(float(jax_det_stat(data_sfts, fa, fda, T_sft=T_sft)))
    except Exception:
        stats_random.append(0.0)

print(f'Random draws : mean stat = {np.mean(stats_random):.2f}, max = {np.max(stats_random):.2f}')

methods = {
    'Random (n=20)'  : (0.0,   float(np.max(stats_random))),
    'TrackOptimizer' : (opt_time, stat_best),
    'PSO (n=50,20it)': (pso_time, -pso_fitness),
    'Truth upper bound': (0.0, stat_true),
}

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['C3', 'C0', 'C1', 'k']
markers = ['x', 'o', 's', '*']
for (name, (t_s, s)), c, mk in zip(methods.items(), colors, markers):
    ax.scatter(t_s, s, c=c, marker=mk, s=120, label=f'{name}: stat={s:.1f}')

ax.set_xlabel('Wall-clock time [s]')
ax.set_ylabel('Detection statistic')
ax.set_title('Recovery comparison')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()


## 7. (Optional) Gradient-Based Refinement with fewtrax

If `fewtrax` is installed (from `JAX-waveform/fewtrax/`), this cell demonstrates
gradient-based track optimization using the `TrackOptimizerJAX` class.

```bash
pip install /path/to/JAX-waveform/fewtrax
# or add to path:
import sys; sys.path.insert(0, '/path/to/JAX-waveform/fewtrax/src')
```

In [ ]:
import os, sys

FEWTRAX_SRC = '/Users/bertd/Documents/PhD/LISA/Projects/JAX-waveform/fewtrax/src'
FEW_DATA_DIR = os.environ.get('FEW_DATA_DIR')

if os.path.isdir(FEWTRAX_SRC) and FEW_DATA_DIR:
    sys.path.insert(0, FEWTRAX_SRC)
    try:
        from fewtrax.data.loader import load_flux_data
        from emrisearch.track_optimizer import TrackOptimizerJAX

        flux_data = load_flux_data(FEW_DATA_DIR)
        jax_opt = TrackOptimizerJAX(f_obs, fdot_obs, t_obs, flux_data, T_sft=T_sft)

        print(f'T_plunge estimate: {jax_opt.T_plunge_estimate:.4f} yr')

        t0 = time.perf_counter()
        jax_params, jax_loss = jax_opt.optimize_full(
            mode=(m, 0, n), n_starts=8, adam_steps=150, lbfgs_iter=100
        )
        jax_time = time.perf_counter() - t0

        print(f'\nTrackOptimizerJAX (Adam + L-BFGS):')
        print(f'  Time      : {jax_time:.1f} s')
        print(f'  Track loss: {jax_loss:.3e}')
        print(f'  Params    : M={jax_params[0]:.3e}, mu={jax_params[1]:.2f}, '
              f'a={jax_params[2]:.3f}, Tpl={jax_params[3]:.4f}, ef={jax_params[4]:.4f}')

        # Fisher information matrix
        F = jax_opt.compute_fisher(jax_params, mode=(m, 0, n))
        cov = np.linalg.inv(F)
        sigmas = np.sqrt(np.diag(cov))
        param_names = ['M', 'mu', 'a', 'T_plunge', 'e_f']
        print('\nFisher Cramér-Rao bounds (1-sigma):')
        for nm, s in zip(param_names, sigmas):
            print(f'  σ({nm}) = {s:.3e}')

    except Exception as e:
        print(f'fewtrax optimization failed: {e}')
else:
    print('fewtrax not found or FEW_DATA_DIR not set; skipping gradient demo.')
    print(f'  FEWTRAX_SRC exists: {os.path.isdir(FEWTRAX_SRC)}')
    print(f'  FEW_DATA_DIR      : {FEW_DATA_DIR}')


## Summary

| Method | Backend | Key advantage |
|--------|---------|---------------|
| `TrackOptimizer` | FEW (numpy) | No extra deps; T_plunge-focused init |
| `ParticleSwarmOptimizer` | FEW (numpy) | Global search; robust to multimodality |
| `TrackOptimizerJAX` | fewtrax (JAX) | Full gradients; O(50×) fewer steps |
| `TrackOptimizerJAX.compute_fisher` | fewtrax | Cramér-Rao parameter precision |

**Recommended workflow:**
1. Run PSO (200 particles, 100 iter) for global exploration
2. Pass top-5 PSO candidates to `TrackOptimizer.optimize()` for refinement
3. If fewtrax available: run `TrackOptimizerJAX.optimize_full()` from best candidate
4. Compute Fisher matrix to assess parameter recovery quality
5. Run `mcmc_followup.py` for full Bayesian posterior
